<a href="https://colab.research.google.com/github/jetnipitbank-maker/thai-bank-sentiment-analysis/blob/main/07_analyze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install pandas openpyxl numpy statsmodels linearmodels

In [ ]:
# [AGGREGATION - FinBERT QUANT STYLE]
import pandas as pd
import os
import glob

# กำหนด Path โฟลเดอร์ที่แยก 6 ธนาคารไว้แล้ว
input_folder = r"C:\Users\USER\Desktop\Banks_Separated_News"
output_folder = r"C:\Users\USER\Desktop\Banks_Daily_Sentiment_Panel"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

file_list = glob.glob(os.path.join(input_folder, "*_Labeled_News.csv"))
print("🔄 เริ่มกระบวนการยุบรวมข้อมูลเป็นรายวัน (Daily Aggregation)...")

for file in file_list:
    bank_name = os.path.basename(file).split('_')[0]
    df = pd.read_csv(file)
    df['date'] = pd.to_datetime(df['date'])

    # ยุบรวมข้อมูลรายวัน
    df_daily = df.groupby('date').agg(
        total_news = ('polarity_score', 'count'),          # 1. จำนวนข่าวทั้งหมด
        mean_polarity = ('polarity_score', 'mean'),        # 2. ค่าเฉลี่ยอารมณ์รวมของวันนั้น
        pos_count = ('sentiment_label', lambda x: (x == 'Positive').sum()), # 3. นับจำนวนข่าวบวก
        neg_count = ('sentiment_label', lambda x: (x == 'Negative').sum()), # 4. นับจำนวนข่าวลบ
        neu_count = ('sentiment_label', lambda x: (x == 'Neutral').sum())
    ).reset_index()

    # 🌟 สร้างตัวแปร Daily Sentiment Spread (สูตรฮิตในเปเปอร์ Quant)
    # (จำนวนข่าวบวก - จำนวนข่าวลบ) / จำนวนข่าวทั้งหมด
    df_daily['sentiment_spread'] = (df_daily['pos_count'] - df_daily['neg_count']) / df_daily['total_news']

    # ปัดเศษ
    df_daily['mean_polarity'] = df_daily['mean_polarity'].round(4)
    df_daily['sentiment_spread'] = df_daily['sentiment_spread'].round(4)

    # ใส่ชื่อธนาคารเพื่อเตรียมทำ Panel Data
    df_daily.insert(1, 'bank_symbol', bank_name)

    # บันทึกไฟล์
    output_filename = os.path.join(output_folder, f"{bank_name}_Daily_Sentiment.csv")
    df_daily.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"   ✅ {bank_name}: ได้ข้อมูล Panel {len(df_daily):,} วัน")

print(f"\n🎉 สำเร็จ! ไฟล์พร้อมนำไปรัน Regression กับราคาหุ้นแล้วครับ")

🔄 เริ่มกระบวนการยุบรวมข้อมูลเป็นรายวัน (Daily Aggregation)...
   ✅ BAY: ได้ข้อมูล Panel 198 วัน
   ✅ BBL: ได้ข้อมูล Panel 222 วัน
   ✅ KBANK: ได้ข้อมูล Panel 271 วัน
   ✅ KTB: ได้ข้อมูล Panel 261 วัน
   ✅ SCB: ได้ข้อมูล Panel 245 วัน
   ✅ TTB: ได้ข้อมูล Panel 203 วัน

🎉 สำเร็จ! ไฟล์พร้อมนำไปรัน Regression กับราคาหุ้นแล้วครับ


In [ ]:
# [PART 3 - THAI HEADERS & SEPARATED FILES: Data Merging]
import pandas as pd
import os
import glob
import numpy as np

# =======================================================
# 1. กำหนด Path โฟลเดอร์
# =======================================================
sentiment_folder = r"C:\Users\USER\Desktop\Banks_Daily_Sentiment_Panel"
stock_folder = r"C:\Users\USER\Desktop\Data_SET\BANKS_DATA_2025"
output_folder = r"C:\Users\USER\Desktop\Master_Panel_Data"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

print("🔄 1. กำลังเตรียมกระบวนการ Merge ข้อมูล Sentiment กับ Stock Prices...")

target_banks = ['KBANK', 'SCB', 'BBL', 'KTB', 'BAY', 'TTB']
all_panel_data = []

# =======================================================
# 2. วนลูปทำงานทีละธนาคาร
# =======================================================
for bank in target_banks:
    print(f"\nกำลังจัดการข้อมูลของ: {bank}")

    # ---------------------------------------------------
    # 2.1 โหลดข้อมูล Sentiment
    # ---------------------------------------------------
    sentiment_file = os.path.join(sentiment_folder, f"{bank}_Daily_Sentiment.csv")
    if not os.path.exists(sentiment_file):
        print(f"   ⚠️ ไม่พบไฟล์ Sentiment ข้ามไปก่อน")
        continue

    df_sent = pd.read_csv(sentiment_file)
    df_sent['date'] = pd.to_datetime(df_sent['date']).dt.strftime('%Y-%m-%d')

    # ---------------------------------------------------
    # 2.2 โหลดข้อมูลราคาหุ้น (ค้นหาไฟล์ที่มีชื่อแบงก์ในโฟลเดอร์)
    # ---------------------------------------------------
    # รองรับทั้งนามสกุล .csv และ .xlsx
    stock_files = glob.glob(os.path.join(stock_folder, f"*{bank}*.csv")) + \
                  glob.glob(os.path.join(stock_folder, f"*{bank}*.xlsx"))

    if not stock_files:
        print(f"   ⚠️ ไม่พบไฟล์ราคาหุ้นของ {bank} ในโฟลเดอร์")
        continue

    stock_file_path = stock_files[0] # เลือกไฟล์แรกที่เจอ

    # อ่านไฟล์ (ลองใช้ utf-8 ก่อน ถ้าพังให้ใช้ tis-620 ซึ่งเป็นฟอร์แมตภาษาไทย)
    try:
        if stock_file_path.endswith('.csv'):
            df_price = pd.read_csv(stock_file_path, encoding='utf-8-sig')
        else:
            df_price = pd.read_excel(stock_file_path)
    except UnicodeDecodeError:
        df_price = pd.read_csv(stock_file_path, encoding='tis-620')

    # ---------------------------------------------------
    # 2.3 เปลี่ยนชื่อหัวคอลัมน์ภาษาไทย & ทำความสะอาดตัวเลข
    # ---------------------------------------------------
    rename_columns = {
        'วันที่': 'date',
        'ราคาเปิด': 'Open',
        'ราคาสูงสุด': 'High',
        'ราคาต่ำสุด': 'Low',
        'ราคาปิด': 'Close',
        'ปริมาณซื้อขาย (พันหุ้น)': 'Volume'
    }
    df_price.rename(columns=rename_columns, inplace=True)

    # แปลงวันที่ให้เป็น Format มาตรฐาน (YYYY-MM-DD)
    # ใช้ dayfirst=True เพราะข้อมูลไทยมักมาแบบ วว/ดด/ปปปป
    df_price['date'] = pd.to_datetime(df_price['date'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')

    # ล้างลูกน้ำ (,) และแปลงตัวเลขที่อาจมาเป็น Text ให้กลายเป็น Float
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    for col in numeric_cols:
        if col in df_price.columns:
            df_price[col] = df_price[col].astype(str).str.replace(',', '', regex=False)
            df_price[col] = pd.to_numeric(df_price[col], errors='coerce')

    # ลบแถวที่ราคาปิดเป็นค่าว่าง (เช่น วันหยุดนักขัตฤกษ์ที่อาจหลุดมา)
    df_price = df_price.dropna(subset=['Close']).sort_values(by='date').reset_index(drop=True)

    # ---------------------------------------------------
    # 2.4 คำนวณตัวแปรทางสถิติ (Return & Volatility)
    # ---------------------------------------------------
    df_price['daily_return'] = df_price['Close'].pct_change()
    df_price['volatility'] = (df_price['High'] - df_price['Low']) / df_price['Open']

    # ---------------------------------------------------
    # 2.5 ประกบร่าง (Merge)
    # ---------------------------------------------------
    df_merged = pd.merge(df_price, df_sent, on='date', how='left')

    # เติม 0 สำหรับวันที่ไม่มีข่าว
    cols_to_fill = ['total_news', 'mean_polarity', 'pos_count', 'neg_count', 'neu_count', 'sentiment_spread']
    for c in cols_to_fill:
        if c in df_merged.columns:
            df_merged[c] = df_merged[c].fillna(0)

    # 🌟 [แก้ไขตรงนี้] อัปเดต/สร้างคอลัมน์ชื่อธนาคารแบบบังคับทับลงไปเลย
    df_merged['bank_symbol'] = bank

    # ทิ้งวันแรกสุดที่หา Return ไม่ได้
    df_merged = df_merged.dropna(subset=['daily_return']).reset_index(drop=True)

    # เลือกคอลัมน์สำคัญไปใช้งาน
    cols_order = ['date', 'bank_symbol', 'Open', 'High', 'Low', 'Close', 'Volume',
                  'daily_return', 'volatility',
                  'total_news', 'mean_polarity', 'sentiment_spread', 'pos_count', 'neg_count']
    final_cols = [c for c in cols_order if c in df_merged.columns]
    df_merged = df_merged[final_cols]

    # เซฟแยกรายธนาคาร
    indv_output = os.path.join(output_folder, f"{bank}_Master_Data.csv")
    df_merged.to_csv(indv_output, index=False, encoding='utf-8-sig')
    print(f"   ✅ {bank}: ประกอบร่างสำเร็จ ({len(df_merged)} วันทำการ)")

    all_panel_data.append(df_merged)

# =======================================================
# 3. สร้าง Master Panel Data (รวมทุกธนาคารในไฟล์เดียว)
# =======================================================
if all_panel_data:
    df_master_panel = pd.concat(all_panel_data, ignore_index=True)
    master_output = os.path.join(output_folder, "FINAL_MASTER_PANEL_DATA.csv")
    df_master_panel.to_csv(master_output, index=False, encoding='utf-8-sig')

    print("\n" + "="*60)
    print("🎉 สร้างข้อมูล Master Panel Data เสร็จสมบูรณ์!")
    print(f"📁 เปิดไฟล์รวม 6 ธนาคารได้ที่: {master_output}")
    print("="*60)

🔄 1. กำลังเตรียมกระบวนการ Merge ข้อมูล Sentiment กับ Stock Prices...

กำลังจัดการข้อมูลของ: KBANK
   ✅ KBANK: ประกอบร่างสำเร็จ (241 วันทำการ)

กำลังจัดการข้อมูลของ: SCB
   ✅ SCB: ประกอบร่างสำเร็จ (241 วันทำการ)

กำลังจัดการข้อมูลของ: BBL
   ✅ BBL: ประกอบร่างสำเร็จ (241 วันทำการ)

กำลังจัดการข้อมูลของ: KTB
   ✅ KTB: ประกอบร่างสำเร็จ (241 วันทำการ)

กำลังจัดการข้อมูลของ: BAY
   ✅ BAY: ประกอบร่างสำเร็จ (241 วันทำการ)

กำลังจัดการข้อมูลของ: TTB
   ✅ TTB: ประกอบร่างสำเร็จ (241 วันทำการ)

🎉 สร้างข้อมูล Master Panel Data เสร็จสมบูรณ์!
📁 เปิดไฟล์รวม 6 ธนาคารได้ที่: C:\Users\USER\Desktop\Master_Panel_Data\FINAL_MASTER_PANEL_DATA.csv


In [ ]:
import pandas as pd

# ==============================================
# โหลด SET Index และแปลงวันที่ พ.ศ. → ค.ศ.
# ==============================================
set_tables = pd.read_html(
    r"C:\Users\USER\Desktop\Data_SET\sectoralDataPrice_excel.xls",
    encoding='utf-8'
)
set_df = set_tables[5].copy()
set_df.columns = set_df.iloc[1]
set_df = set_df.iloc[2:].reset_index(drop=True)
set_df = set_df[['วันที่', '%เปลี่ยนแปลง']].copy()
set_df.columns = ['date_raw', 'market_return']
set_df = set_df.dropna(subset=['date_raw'])

# ✅ แปลงปี พ.ศ. → ค.ศ. ถูกต้อง
def convert_thai_date(date_str):
    try:
        d, m, y = str(date_str).strip().split('/')
        y_ad = int(y) - 543          # พ.ศ. → ค.ศ.
        return pd.Timestamp(f"{y_ad}-{m}-{d}")
    except:
        return pd.NaT

set_df['date'] = set_df['date_raw'].apply(convert_thai_date)
set_df['market_return'] = (
    pd.to_numeric(set_df['market_return'], errors='coerce') / 100
)
set_df = set_df[['date', 'market_return']].dropna()

print(f"✅ SET Index: {len(set_df)} วัน")
print(set_df.head())
# ควรเห็น: 2025-01-02, 2025-01-03, ...

# ==============================================
# Merge เข้า Panel Data
# ==============================================
df = pd.read_csv(r"C:\Users\USER\Desktop\Master_Panel_Data\FINAL_MASTER_PANEL_DATA.csv")
df['date'] = pd.to_datetime(df['date'])

df = df.merge(set_df, on='date', how='left')

missing = df['market_return'].isna().sum()
print(f"\n⚠️  market_return ขาด: {missing} แถว")
print(f"✅ market_return มีค่า: {df['market_return'].notna().sum()} แถว")
print(f"\nตัวอย่างข้อมูลหลัง merge:")
print(df[['date', 'bank_symbol', 'market_return']].head(10))

# ==============================================
# คำสั่งสำหรับ Save ไฟล์
# ==============================================
# กำหนด path ที่ต้องการบันทึก (แนะนำให้ตั้งชื่อไฟล์ใหม่เพื่อกันการเซฟทับไฟล์ต้นฉบับผิดพลาด)
save_path = r"C:\Users\USER\Desktop\Master_Panel_Data\FINAL_MASTER_PANEL_DATA_MERGED.csv"

# บันทึกเป็นไฟล์ CSV
df.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"\n💾 บันทึกไฟล์ข้อมูลสำเร็จ! ตรวจสอบไฟล์ได้ที่: {save_path}")

✅ SET Index: 242 วัน
        date  market_return
0 2025-12-30         0.0045
1 2025-12-29        -0.0041
2 2025-12-26        -0.0044
3 2025-12-25        -0.0083
4 2025-12-24         0.0033

⚠️  market_return ขาด: 0 แถว
✅ market_return มีค่า: 1446 แถว

ตัวอย่างข้อมูลหลัง merge:
        date bank_symbol  market_return
0 2025-01-03       KBANK         0.0036
1 2025-01-06       KBANK        -0.0087
2 2025-01-07       KBANK         0.0133
3 2025-01-08       KBANK        -0.0023
4 2025-01-09       KBANK        -0.0178
5 2025-01-10       KBANK         0.0037
6 2025-01-13       KBANK        -0.0100
7 2025-01-14       KBANK        -0.0104
8 2025-01-15       KBANK         0.0096
9 2025-01-16       KBANK        -0.0005

💾 บันทึกไฟล์ข้อมูลสำเร็จ! ตรวจสอบไฟล์ได้ที่: C:\Users\USER\Desktop\Master_Panel_Data\FINAL_MASTER_PANEL_DATA_MERGED.csv


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from IPython.display import display, HTML

# =======================================================
# ⚙️ STEP 0: ตั้งค่า Path ไฟล์
# =======================================================
PANEL_DATA_PATH = r"C:\Users\USER\Desktop\Master_Panel_Data\FINAL_MASTER_PANEL_DATA_MERGED.csv"

# =======================================================
# 📥 STEP 1: โหลด Panel Data (market_return รวมอยู่แล้ว)
# =======================================================
try:
    df = pd.read_csv(PANEL_DATA_PATH)
    df['date'] = pd.to_datetime(df['date'])

    panel_data = df.set_index(['bank_symbol', 'date']).sort_index()
    panel_data = panel_data.replace([float('inf'), float('-inf')], pd.NA)

    # คำนวณ sentiment_spread ถ้ายังไม่มีในไฟล์
    if 'sentiment_spread' not in panel_data.columns:
        panel_data['sentiment_spread'] = (
            (panel_data['pos_count'] - panel_data['neg_count'])
            / panel_data['total_news']
        ).round(4)
        print("⚙️  คำนวณ sentiment_spread จาก pos/neg/total แล้ว")

    # Drop rows ที่ขาดตัวแปรจำเป็น
    required = ['mean_polarity', 'sentiment_spread',
                'market_return', 'Volume', 'volatility']
    panel_data = panel_data.dropna(subset=required)

    # Time-Lagged Returns
    for lag in range(4):
        panel_data[f'return_T{lag}'] = (
            panel_data.groupby(level='bank_symbol')['daily_return'].shift(-lag)
        )

    banks = panel_data.index.get_level_values('bank_symbol').unique()
    print(f"✅ โหลดข้อมูลสำเร็จ | ธนาคาร: {sorted(banks.tolist())}")
    print(f"   จำนวนแถว: {len(panel_data)}")
    print(f"   ช่วงข้อมูล: {panel_data.index.get_level_values('date').min().date()} "
          f"ถึง {panel_data.index.get_level_values('date').max().date()}\n")

except Exception as e:
    print(f"❌ โหลดข้อมูลไม่สำเร็จ: {e}"); raise

# =======================================================
# 📈 STEP 2: OLS Regression — 3 โมเดล
#
# โมเดล 1 — Baseline (Unconditional):
#   R_i,t = β0 + β1·mean_polarity + ε
#
# โมเดล 2 — Main Model:
#   R_i,t = β0 + β1·mean_polarity + β2·R_m,t
#           + β3·Volume + β4·Volatility + ε
#
# โมเดล 3 — Robustness Check:
#   R_i,t = β0 + β1·sentiment_spread + β2·R_m,t
#           + β3·Volume + β4·Volatility + ε
# =======================================================

# แต่ละโมเดล = (sentiment_var, control_vars)
MODELS = {
    'Baseline Model — Unconditional (Mean Polarity)':
        ('mean_polarity',    []),
    'Main Model — With Controls (Mean Polarity)':
        ('mean_polarity',    ['market_return', 'Volume', 'volatility']),
    'Robustness Check — With Controls (Sentiment Spread)':
        ('sentiment_spread', ['market_return', 'Volume', 'volatility']),
}

LAGS = {
    'Day T (0)': 'return_T0',
    'Day T+1':   'return_T1',
    'Day T+2':   'return_T2',
    'Day T+3':   'return_T3',
}

# สไตล์
BG, FG, LINE   = '#282a2d', '#ffffff', '#8e8e8e'
HEAD_BASELINE  = '#2a1a2a'   # ม่วงเข้ม  — Baseline
HEAD_MAIN      = '#1a3a2a'   # เขียวเข้ม — Main
HEAD_ROBUST    = '#2a2a1a'   # เหลืองเข้ม — Robustness

HEAD_COLORS = {
    'Baseline Model — Unconditional (Mean Polarity)':          HEAD_BASELINE,
    'Main Model — With Controls (Mean Polarity)':              HEAD_MAIN,
    'Robustness Check — With Controls (Sentiment Spread)':     HEAD_ROBUST,
}

def run_ols(df_bank, sentiment_var, control_vars, lag_col):
    """รัน OLS 1 สมการ — รองรับทั้งแบบมีและไม่มี control vars"""
    all_vars = [lag_col, sentiment_var] + control_vars
    data = df_bank[all_vars].dropna()

    if len(data) < 10:
        return {'β₁': 'N/A', 'P-Value': 'N/A',
                'Sig.': '', 'R²': 'N/A', 'Obs.': len(data)}

    Y = data[lag_col]
    X = sm.add_constant(data[[sentiment_var] + control_vars])
    model = sm.OLS(Y, X).fit(cov_type='HC1')

    coef = model.params[sentiment_var]
    pval = model.pvalues[sentiment_var]
    sig  = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''

    return {
        'β₁':      f"{coef:.5f}",
        'P-Value': f"{pval:.4f}",
        'Sig.':    sig,
        'R²':      f"{model.rsquared:.4f}",
        'Obs.':    len(data)
    }

def make_table_html(df_res, header_color, model_label):
    """สร้าง HTML ตาราง — คอลัมน์ตรงกับหัวตาราง"""
    col_styles = [
        "width:120px;text-align:left;font-weight:bold",
        "width:110px;text-align:center",
        "width:110px;text-align:center",
        "width:80px;text-align:center",
        "width:100px;text-align:center",
        "width:80px;text-align:center",
    ]
    headers = ['Time Lag', 'β₁', 'P-Value', 'Sig.', 'R²', 'Obs.']

    th = ''.join(
        f"<th style='padding:10px 12px;border-bottom:1px solid {LINE};"
        f"text-align:center;{col_styles[i]}'>{h}</th>"
        for i, h in enumerate(headers)
    )

    rows_html = ''
    for _, row in df_res.iterrows():
        sig_color = ('#4CAF50' if '***' in str(row['Sig.'])
                     else '#FFC107' if '**'  in str(row['Sig.'])
                     else '#FF9800' if '*'   in str(row['Sig.'])
                     else FG)
        vals = [row['Time Lag'], row['β₁'], row['P-Value'],
                row['Sig.'], row['R²'], row['Obs.']]
        tds = ''
        for i, v in enumerate(vals):
            color = sig_color if i == 3 else FG
            tds += (f"<td style='{col_styles[i]};padding:8px 12px;"
                    f"border-bottom:1px solid #3d3d3d;color:{color};"
                    f"font-weight:{'bold' if i==3 else 'normal'}'>{v}</td>")
        rows_html += f"<tr>{tds}</tr>"

    return f"""
    <table style='background:{BG};color:{FG};font-family:Segoe UI,sans-serif;
                  border-collapse:collapse;table-layout:fixed;
                  width:620px;margin-bottom:8px;'>
      <thead>
        <tr style='background:{header_color};'>
          <th colspan='6' style='padding:8px 12px;text-align:left;
                                  color:{FG};font-size:13px;
                                  border-top:2px solid {LINE};
                                  border-bottom:1px solid {LINE};'>
            {model_label}
          </th>
        </tr>
        <tr style='background:#1e1f22;'>{th}</tr>
      </thead>
      <tbody>{rows_html}</tbody>
    </table>"""

# =======================================================
# 🔄 STEP 3: วนลูปแยกรายธนาคาร
# =======================================================
for bank in sorted(banks):
    df_bank = panel_data.xs(bank, level='bank_symbol')

    display(HTML(
        f"<h3 style='color:#4CAF50;font-family:Segoe UI,sans-serif;"
        f"margin:20px 0 5px;'>🏦 {bank}</h3>"
    ))

    for model_label, (sentiment_var, control_vars) in MODELS.items():
        results = []
        for lag_label, lag_col in LAGS.items():
            res = run_ols(df_bank, sentiment_var, control_vars, lag_col)
            res['Time Lag'] = lag_label
            results.append(res)

        df_res = pd.DataFrame(results)[
            ['Time Lag', 'β₁', 'P-Value', 'Sig.', 'R²', 'Obs.']
        ]
        display(HTML(make_table_html(df_res, HEAD_COLORS[model_label], model_label)))

# =======================================================
# 📝 หมายเหตุท้าย
# =======================================================
display(HTML(f"""
<div style='color:{LINE};font-size:13px;line-height:2.0;
            margin-top:15px;font-family:Segoe UI,sans-serif;
            border-top:1px solid #3d3d3d;padding-top:10px;'>
  <b>โมเดล 1 — Baseline:</b>
    R<sub>i,t</sub> = β₀ + β₁·<b>MeanPolarity</b><sub>i,t</sub> + ε<sub>i,t</sub><br>
  <b>โมเดล 2 — Main Model:</b>
    R<sub>i,t</sub> = β₀ + β₁·<b>MeanPolarity</b><sub>i,t</sub>
    + β₂·R<sub>m,t</sub> + β₃·Volume<sub>i,t</sub>
    + β₄·Volatility<sub>i,t</sub> + ε<sub>i,t</sub><br>
  <b>โมเดล 3 — Robustness Check:</b>
    R<sub>i,t</sub> = β₀ + β₁·<b>SentimentSpread</b><sub>i,t</sub>
    + β₂·R<sub>m,t</sub> + β₃·Volume<sub>i,t</sub>
    + β₄·Volatility<sub>i,t</sub> + ε<sub>i,t</sub><br>
  <b>ระดับนัยสำคัญ:</b>
    <span style='color:#4CAF50'>***</span> p&lt;0.01 &nbsp;
    <span style='color:#FFC107'>**</span> p&lt;0.05 &nbsp;
    <span style='color:#FF9800'>*</span> p&lt;0.1<br>
  <b>หมายเหตุ:</b> β₁ แสดงเฉพาะอิทธิพลของ Sentiment | Robust SE: HC1
</div>
"""))

✅ โหลดข้อมูลสำเร็จ | ธนาคาร: ['BAY', 'BBL', 'KBANK', 'KTB', 'SCB', 'TTB']
   จำนวนแถว: 1446
   ช่วงข้อมูล: 2025-01-03 ถึง 2025-12-30

